# shira_motion02 v3.1 — Wan 2.2 I2V one-shot. Frame: x0.at/puPP.jpg | 81f | seed 602 | shimi-pituach for shimi-creative

In [ ]:
import torch, os, sys, subprocess, time, requests, shutil, traceback, json as _j
LOG = []
def log(m):
    LOG.append(str(m)); print('STAR ' + str(m), flush=True)
def save_runlog():
    try:
        tail = open('/kaggle/temp/comfy.log').read()[-1500:]
        LOG.append('--- comfy.log tail ---'); LOG.append(tail)
    except Exception: pass
    open('/kaggle/working/run.log','w').write(chr(10).join(LOG))
    print('runlog written', flush=True)

COMFY = '/kaggle/temp/ComfyUI'
try:
    log('GPU: ' + (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'))
    log('VRAM GB: ' + (str(round(torch.cuda.get_device_properties(0).total_memory/1e9,1)) if torch.cuda.is_available() else '0'))
    os.makedirs('/kaggle/temp', exist_ok=True)
    CUSTOM = COMFY + '/custom_nodes'
    if not os.path.exists(COMFY):
        log('install ComfyUI')
        subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git',COMFY], check=True)
    os.chdir(COMFY)
    subprocess.run(['pip','install','-q','-r','requirements.txt'], check=True)
    for nm, url in {'ComfyUI-VideoHelperSuite':'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git','ComfyUI-GGUF':'https://github.com/city96/ComfyUI-GGUF.git'}.items():
        if not os.path.exists(CUSTOM + '/' + nm):
            log('node ' + nm)
            subprocess.run(['git','clone','--depth','1',url,CUSTOM + '/' + nm], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','gguf'], check=True)
    log('gguf module installed')
    subprocess.run(['pip','install','-q','imageio-ffmpeg','imageio[ffmpeg]'], check=True)
    log('ComfyUI ready')

    M = COMFY + '/models'
    files = [
        (M+'/diffusion_models','Wan2.2-TI2V-5B-Q4_K_M.gguf','https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/Wan2.2-TI2V-5B-Q4_K_M.gguf'),
        (M+'/text_encoders','umt5_xxl_fp8_e4m3fn_scaled.safetensors','https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors'),
        (M+'/vae','Wan2.2_VAE.safetensors','https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/VAE/Wan2.2_VAE.safetensors'),
    ]
    for d, fn, url in files:
        os.makedirs(d, exist_ok=True)
        dest = d + '/' + fn
        if not os.path.exists(dest):
            log('download ' + fn)
            r = requests.get(url, stream=True, timeout=600); r.raise_for_status()
            with open(dest,'wb') as f:
                for c in r.iter_content(1024*1024): f.write(c)
        log('ok ' + fn + ' ' + str(round(os.path.getsize(dest)/1e9,2)) + 'GB')
    os.makedirs(COMFY+'/input', exist_ok=True)
    r = requests.get('https://x0.at/puPP.jpg', timeout=60); r.raise_for_status()
    open(COMFY+'/input/shira_frame.jpg','wb').write(r.content)
    log('shira frame: ' + str(len(r.content)) + ' bytes')

    def start(extra):
        args = ['python','main.py','--listen','127.0.0.1','--port','8188','--dont-print-server','--disable-auto-launch','--lowvram'] + extra
        p = subprocess.Popen(args, cwd=COMFY, stdout=open('/kaggle/temp/comfy.log','w'), stderr=subprocess.STDOUT)
        for _ in range(120):
            time.sleep(5)
            try:
                if requests.get('http://127.0.0.1:8188/system_stats', timeout=3).status_code == 200:
                    log('ComfyUI up ' + str(extra)); return p
            except Exception: pass
            if p.poll() is not None:
                log('ComfyUI died rc=' + str(p.returncode)); return None
        return None
    proc = start([])
    if proc is None:
        proc = start(['--novram'])
    if proc is None:
        raise RuntimeError('ComfyUI failed: ' + open('/kaggle/temp/comfy.log').read()[-1200:])

    PROMPT = "After this frame: the young woman squeezes the piping bag with steady hand pressure, a continuous ribbon of cream lands on the cake top in a smooth spiral, her wrist rotates with the flow, her second hand steadies the base of the bag, shoulders stay level, the head tilts slightly down to follow the line being piped, the cake turns slowly on its stand under her guiding hand. Static camera on a tripod, fixed kitchen lighting, background unchanged, realistic body physics, no camera movement, no zoom."
    NEG = "blurry, low quality, pixelated, interpolated image, plastic texture, jpeg artifacts, fast motion, slow motion, glitch, deformed body, deformed hands, extra fingers, worst quality, bad quality, overexposed, watermark, text"
    W, H, FRAMES, FPS, SEED = 480, 608, 81, 16, 602
    wf = {
     "1":  {"class_type":"UnetLoaderGGUF","inputs":{"unet_name":"Wan2.2-TI2V-5B-Q4_K_M.gguf"}},
     "2":  {"class_type":"CLIPLoader","inputs":{"clip_name":"umt5_xxl_fp8_e4m3fn_scaled.safetensors","type":"wan"}},
     "3":  {"class_type":"ModelSamplingSD3","inputs":{"model":["1",0],"shift":8.0}},
     "4":  {"class_type":"CLIPTextEncode","inputs":{"text":NEG,"clip":["2",0]}},
     "5":  {"class_type":"CLIPTextEncode","inputs":{"text":PROMPT,"clip":["2",0]}},
     "8":  {"class_type":"VAELoader","inputs":{"vae_name":"Wan2.2_VAE.safetensors"}},
     "12": {"class_type":"LoadImage","inputs":{"image":"shira_frame.jpg"}},
     "14": {"class_type":"WanImageToVideo","inputs":{"positive":["5",0],"negative":["4",0],"vae":["8",0],"width":W,"height":H,"length":FRAMES,"batch_size":1,"start_image":["12",0]}},
     "7":  {"class_type":"KSampler","inputs":{"seed":SEED,"steps":20,"cfg":5.0,"sampler_name":"euler","scheduler":"simple","denoise":1.0,"model":["3",0],"positive":["14",0],"negative":["14",1],"latent_image":["14",2]}},
     "9":  {"class_type":"VAEDecode","inputs":{"samples":["7",0],"vae":["8",0]}},
     "10": {"class_type":"VHS_VideoCombine","inputs":{"frame_rate":FPS,"loop_count":0,"pingpong":False,"format":"video/h264-mp4","filename_prefix":"shira_motion02","save_output":True,"images":["9",0]}},
    }
    r = requests.post('http://127.0.0.1:8188/prompt', json={"prompt":wf,"client_id":"shimi"}, timeout=60)
    if r.status_code >= 400:
        raise RuntimeError('submit error: ' + r.text[:800])
    pid = r.json()['prompt_id']
    log('render started ' + pid)
    t0 = time.time(); done = False
    while not done:
        time.sleep(30)
        try:
            h = requests.get('http://127.0.0.1:8188/history/' + pid, timeout=10).json()
        except Exception as e:
            log('history err ' + str(e)); continue
        if pid in h:
            st = h[pid].get('status', {})
            if st.get('status_str') == 'error':
                raise RuntimeError('ComfyUI error: ' + str(st.get('messages'))[:900])
            for nid, out in h[pid].get('outputs', {}).items():
                for g in (out.get('gifs') or out.get('videos') or []):
                    src = COMFY + '/output/' + g['filename']
                    if os.path.exists(src):
                        shutil.copy(src, '/kaggle/working/shira_motion02_raw.mp4')
                        log('CLIP OK ' + str(round(os.path.getsize('/kaggle/working/shira_motion02_raw.mp4')/1e6,2)) + 'MB in ' + str(round((time.time()-t0)/60,1)) + 'min')
                        done = True
            continue
        log('...running ' + str(round((time.time()-t0)/60,1)) + 'min')
        if (time.time()-t0)/60 > 150:
            raise TimeoutError('150min')
    _j.dump({'tag':'shira_motion02','status':'DONE_OK','engine':'Wan2.2-TI2V-5B-GGUF-Q4','frames':FRAMES,'fps':FPS,'size':[W,H],'seed':SEED,'dna':{'producer':'shimi-pituach for shimi-creative','character':'shira','playbook':'engine_playbook_v5'}}, open('/kaggle/working/shira_motion02_report.json','w'), ensure_ascii=False, indent=1)
    log('DONE_OK')
except Exception:
    LOG.append('FATAL: ' + traceback.format_exc()[-2000:])
    print('FATAL', flush=True)
finally:
    save_runlog()
